In [ ]:
# ===================================================================================================
# PHASE C. DOES POOLING RAISE THE EXACT-ZERO RATE, OR WAS THAT THE CELL LINE?
#
# WHY THIS RUNS. Phase B3 found the ASC subset had a LOWER exact-zero fraction than the full library
# in both SSc primaries (0.304 vs 0.456; 0.372 vs 0.512) and a HIGHER one in the sensitivity query
# (0.439 vs 0.421). That was written up as evidence that the median-over-signatures rule raises the
# tie rate. THAT COMPARISON IS CONFOUNDED AND THE WRITE-UP HAS BEEN MARKED AS SUCH. The ASC subset
# differs from the full library in TWO ways: one signature per compound, AND one cell line of 58.
#
# THE CONTROL. Draw ONE signature per compound AT RANDOM FROM THE FULL LIBRARY. That is one signature
# per compound with the library's own cell-line composition, so:
#     pooling effect   = tie(median over all signatures) - tie(random single signature)
#     cell line effect = tie(ASC single)                 - tie(random single, same compounds)
#
# THE CRITERION, FIXED HERE BEFORE ANY NUMBER IS COMPUTED (10 queries: the 7 deposited benchmark rows
# plus the 3 SSc-MEK queries). Pooling is held to RAISE the exact-zero
# rate only if, across all queries scored:
#     (1) the pooling effect is POSITIVE in at least 8 of the 10 queries, AND
#     (2) its median across queries is at least +5.0 percentage points.
# Anything less and the B3 observation is attributed to the cell line, Section 4 of the result record
# is withdrawn rather than softened, and there is no benchmark addendum.
#
# AND THE EXPECTATION, STATED IN ADVANCE BECAUSE IT POINTS THE OTHER WAY. B2's own positive control
# scored 3,000 library rows and found 1,641 nonzero, a per-signature exact-zero rate near 45 percent,
# against a pooled Q1 rate of 45.6 percent. THOSE ARE NEARLY EQUAL, which is what NO pooling effect
# looks like. If the criterion fails, that is the result this notebook most expects to produce.
# (Those 3,000 rows are the first 3,000 by gctx column position, not a random sample, so the hint is
# weak. It is recorded so that a failure here cannot be presented afterward as a surprise.)
#
# THIS NOTEBOOK TESTS AN INSTRUMENT, NOT A HYPOTHESIS ABOUT ANY DISEASE. It re-uses the deposited
# benchmark's frozen queries and the SSc-MEK frozen queries as INPUTS. It computes no drug ranking,
# no percentile for any compound, and it cannot change any pre-registered result.
# ===================================================================================================
# CELL 1: load both frozen query sets and the library. Nothing below may change them.
from google.colab import drive
drive.mount("/content/drive")
import os, json, hashlib, time, subprocess, sys, collections
import numpy as np, pandas as pd
BENCH = "/content/drive/MyDrive/RitschelResearch/benchmark"
SSC   = "/content/drive/MyDrive/RitschelResearch/ssc_mek"
W = SSC; os.chdir(W)
BM_SHA  = "1457464f13bac9a7bdeca438f44414f4a2c2a7e0562992375b982217fd5c54e9"
SSC_SHA = "c083e3cfb23cca67b26e772d8435b138d982db02bf74347ad13053c6c42e8301"
REPORT = []
def R(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); REPORT.append(s)

_b = open(os.path.join(BENCH, "signatures_frozen_B1b.json"), "rb").read()
assert hashlib.sha256(_b).hexdigest() == BM_SHA, ("NOT THE DEPOSITED BENCHMARK FREEZE", hashlib.sha256(_b).hexdigest())
BM = json.loads(_b)
_s = open(os.path.join(SSC, "sscmek_signatures_frozen_B1.json"), "rb").read()
assert hashlib.sha256(_s).hexdigest() == SSC_SHA, ("NOT THE CHECKED SSC-MEK FREEZE", hashlib.sha256(_s).hexdigest())
SC = json.loads(_s)

QUERIES = {}
for k, v in BM["signatures"].items():   QUERIES["BM:" + str(k)] = v
for k, v in SC["signatures"].items():   QUERIES["SSC:" + str(k)] = v
QIDS = sorted(QUERIES)
for q in QIDS:
    v = QUERIES[q]
    assert len(v["up"]) == 150 and len(v["dn"]) == 150, ("QUERY SETS MALFORMED", q, len(v["up"]), len(v["dn"]))
    assert not set(v["up"]) & set(v["dn"]), ("UP AND DOWN OVERLAP", q)

g = pd.read_csv("geneinfo_beta.txt", sep="\t")
LMG = g.loc[g.feature_space == "landmark"].copy(); LMG["gene_id"] = LMG.gene_id.astype(int)
assert len(LMG) == 978, ("LANDMARK TABLE IS NOT 978", len(LMG))
M = np.load(os.path.join(BENCH, "libA_matrix.npy"))
perts = np.load(os.path.join(BENCH, "libA_perts.npy"), allow_pickle=True)
genes = np.load(os.path.join(BENCH, "libA_genes.npy")); GI = {int(x): i for i, x in enumerate(genes)}
assert M.shape == (90834, 978) and len(perts) == 90834, M.shape
NEED = 8; FLOOR = 5.0
R("benchmark freeze", BM_SHA)
R("ssc-mek freeze  ", SSC_SHA)
R("library", M.shape, "compounds", len(set(perts.tolist())))
R("queries scored:", len(QIDS), "->", QIDS)
R("CRITERION FIXED BEFORE THE RUN: pooling effect positive in at least %d of %d queries"
  " AND median at least +%.1f points" % (NEED, len(QIDS), FLOOR))
assert len(QIDS) == 10, ("THE CRITERION WAS WRITTEN FOR 10 QUERIES (7 benchmark rows + 3 SSc-MEK)"
                         " AND THIS RUN HAS %d. FIX THE CRITERION DELIBERATELY OR STOP." % len(QIDS))

In [ ]:
# CELL 2: scoring code. ks/conn_all are copied VERBATIM from the deposited benchmark notebook.
# A vectorized form is added for speed and is accepted only after it reproduces conn_all exactly.
def ks(rp, n, t):
    if t == 0 or n == 0: return 0.0
    V = np.sort(np.asarray(rp, float)); j = np.arange(1, t + 1, dtype=float)
    a = np.max(j / t - V / n); b = np.max(V / n - (j - 1.0) / t)
    return float(a) if a > b else float(-b)
def conn_all(Mx, up, dn):
    n = Mx.shape[1]; out = np.empty(Mx.shape[0])
    order = np.argsort(-Mx, axis=1, kind="stable")
    rank = np.empty_like(order); ar = np.arange(1, n + 1)
    np.put_along_axis(rank, order, np.broadcast_to(ar, order.shape), axis=1)
    for i in range(Mx.shape[0]):
        eu = ks(rank[i, up], n, len(up)); ed = ks(rank[i, dn], n, len(dn))
        out[i] = 0.0 if eu * ed >= 0 else (eu - ed) / 2.0
    return out
def rank_matrix(Mx, chunk=8000):
    n = Mx.shape[1]; rk = np.empty(Mx.shape, dtype=np.int32); ar = np.arange(1, n + 1, dtype=np.int32)
    for a in range(0, Mx.shape[0], chunk):
        b = min(a + chunk, Mx.shape[0])
        o = np.argsort(-Mx[a:b], axis=1, kind="stable")
        np.put_along_axis(rk[a:b], o, np.broadcast_to(ar, o.shape), axis=1)
    return rk
def _ksv(sub, n, t):
    V = np.sort(sub, axis=1).astype(np.float64); j = np.arange(1, t + 1, dtype=np.float64)
    a = np.max(j / t - V / n, axis=1); b = np.max(V / n - (j - 1.0) / t, axis=1)
    return np.where(a > b, a, -b)
def conn_from_rank(rk, up, dn):
    n = rk.shape[1]
    eu = _ksv(rk[:, up], n, len(up)); ed = _ksv(rk[:, dn], n, len(dn))
    return np.where(eu * ed >= 0, 0.0, (eu - ed) / 2.0)
def midrank(v):
    v = np.asarray(v, float); n = v.size; o = np.argsort(v, kind="stable"); r = np.empty(n); i = 0
    while i < n:
        j = i
        while j + 1 < n and v[o[j + 1]] == v[o[i]]: j += 1
        r[o[i:j + 1]] = 0.5 * (i + j) + 1.0; i = j + 1
    return r
def spear(a, b):
    ra = midrank(a) - midrank(a).mean(); rb = midrank(b) - midrank(b).mean()
    d = np.sqrt((ra * ra).sum() * (rb * rb).sum())
    return float("nan") if d == 0 else float((ra * rb).sum() / d)
# POSITIVE CONTROL for the vectorized form: it must reproduce conn_all to the bit on a real subset.
_q = QUERIES[QIDS[0]]; _cols = [GI[e] for e in _q["usable"]]; _pos = {e: i for i, e in enumerate(_q["usable"])}
_up = np.array([_pos[e] for e in _q["up"]]); _dn = np.array([_pos[e] for e in _q["dn"]])
_sub = M[:3000][:, _cols]
_a = conn_all(_sub, _up, _dn); _b = conn_from_rank(rank_matrix(_sub), _up, _dn)
assert np.array_equal(_a, _b), ("VECTORIZED SCORE DIFFERS FROM THE DEPOSITED conn_all", float(np.abs(_a - _b).max()))
R("scoring check: the vectorized score reproduces the deposited conn_all exactly on 3,000 signatures")
R("  (nonzero scores in that subset: %d of 3000)" % int((_a != 0).sum()))

In [ ]:
# CELL 3 (unchanged from Phase B3): recover each library row's cell line, and PROVE the recovery is in the library's row order.
# The matrix was built in Phase A by sorting the QC-pass exemplar trt_cp signatures by their column
# position in the LINCS 2020 level-5 gctx. Rebuilding that order from the same two sources must
# reproduce libA_perts.npy element for element, or the cell-line vector is not aligned to the matrix
# and every ASC row below would be the wrong row. This is the same recipe the deposited benchmark
# Phase B3 used; it is re-run here rather than trusted.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fsspec", "aiohttp", "h5py"], check=False)
import fsspec, h5py
t0 = time.time()
s = pd.read_csv(os.path.join(BENCH, "siginfo_beta.txt"), sep="\t", low_memory=False,
                usecols=["sig_id", "pert_id", "pert_type", "is_exemplar_sig", "qc_pass", "cell_iname"])
n_all = len(s)
s = s[(s.pert_type == "trt_cp") & (s.is_exemplar_sig == 1) & (s.qc_pass == 1)]
GCTX = "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/level5/level5_beta_trt_cp_n720216x12328.gctx"
h = h5py.File(fsspec.open(GCTX, "rb").open(), "r")
colid = np.array([x.decode() for x in h["0"]["META"]["COL"]["id"][:]])
pos = {x: i for i, x in enumerate(colid)}
want = sorted((pos[x], p, c) for x, p, c in zip(s.sig_id, s.pert_id, s.cell_iname) if x in pos)
cells = np.array([w[2] for w in want]); p2 = np.array([w[1] for w in want])
assert len(p2) == len(perts), ("ROW COUNT NOT REPRODUCED", len(p2), len(perts))
assert (p2 == perts).all(), ("LIBRARY ROW ORDER NOT REPRODUCED - THE CELL LINE VECTOR IS NOT ALIGNED",
                             int((p2 != perts).sum()))
R("siginfo rows %d -> QC-pass exemplar trt_cp %d | gctx columns %d | %ds"
  % (n_all, len(s), len(colid), round(time.time() - t0)))
R("library row order reproduced from the gctx column ids and siginfo: libA_perts.npy matches element for element")
ASC = cells == "ASC"
R("cell lines recovered: %d distinct | ASC signatures %d of %d rows" % (len(set(cells.tolist())), int(ASC.sum()), len(cells)))
_t = collections.Counter(cells.tolist())
R("the five largest cell lines:", ", ".join("%s %d" % (k, v) for k, v in _t.most_common(5)))
_ap = perts[ASC]
R("ASC compounds %d | signatures per ASC compound: %s"
  % (len(set(_ap.tolist())), ", ".join("%d sig -> %d compounds" % (k, v)
     for k, v in sorted(collections.Counter(collections.Counter(_ap.tolist()).values()).items()))))

In [ ]:
# CELL 4: score every query once on the full library, keeping the per-signature scores. A signature's
# score depends only on its own 978 values and the query, never on which other rows are present, so
# one scoring pass serves all three quantities below.
SCORE = {}; t0 = time.time()
for qid in QIDS:
    q = QUERIES[qid]; cols = [GI[e] for e in q["usable"]]; p_ = {e: i for i, e in enumerate(q["usable"])}
    up = np.array([p_[e] for e in q["up"]]); dn = np.array([p_[e] for e in q["dn"]])
    SCORE[qid] = conn_from_rank(rank_matrix(M[:, cols]), up, dn)
    R("scored %-10s usable genes %d | per-signature exact-zero fraction %.4f | %ds"
      % (qid, len(q["usable"]), float((SCORE[qid] == 0).mean()), round(time.time() - t0)))
# row indices per compound, built once
ORD = np.argsort(perts, kind="stable")
_p = perts[ORD]
_bnd = np.flatnonzero(np.r_[True, _p[1:] != _p[:-1]])
COMPS = _p[_bnd]
ROWS = np.split(ORD, _bnd[1:])
NC = len(COMPS)
assert NC == len(set(perts.tolist())), ("COMPOUND GROUPING IS WRONG", NC)
SIZES = np.array([len(r) for r in ROWS])
assert SIZES.sum() == 90834, SIZES.sum()
ASC_OF = {}
for c, r in zip(COMPS, ROWS):
    a = r[ASC[r]]
    if len(a): ASC_OF[c] = a
R("compound grouping: %d compounds | signatures per compound min %d median %d max %d"
  % (NC, SIZES.min(), int(np.median(SIZES)), SIZES.max()))
R("compounds with at least one ASC signature: %d" % len(ASC_OF))

In [ ]:
# CELL 5: the three quantities. POOLED is the deposited rule. RANDOM-SINGLE is the control: one
# signature per compound drawn from the full library, so cell-line composition is the library's own.
# ASC is one signature per compound from one cell line. The difference POOLED minus RANDOM-SINGLE is
# the pooling effect with cell line held fixed; ASC minus RANDOM-SINGLE on the SAME compounds is the
# cell-line effect with the one-signature design held fixed.
NDRAW = 20
rng = np.random.default_rng(20261002)
ASC_KEYS = sorted(ASC_OF); ASC_SET = set(ASC_KEYS)
ASC_MASKC = np.array([c in ASC_SET for c in COMPS])
TIE = {}
t0 = time.time()
for qid in QIDS:
    sc = SCORE[qid]
    pooled = np.array([float(np.median(sc[r])) for r in ROWS])
    tie_pool_all = float((pooled == 0).mean())
    tie_pool_asc = float((pooled[ASC_MASKC] == 0).mean())
    ra, rs = [], []
    for d in range(NDRAW):
        pick = np.array([r[rng.integers(len(r))] for r in ROWS])
        z = sc[pick] == 0
        ra.append(float(z.mean())); rs.append(float(z[ASC_MASKC].mean()))
    tie_asc = float(np.mean([sc[ASC_OF[c]][0] == 0 for c in ASC_KEYS]))
    TIE[qid] = dict(pooled_all=tie_pool_all, pooled_ascset=tie_pool_asc,
                    rand_all=float(np.mean(ra)), rand_all_sd=float(np.std(ra)),
                    rand_ascset=float(np.mean(rs)), rand_ascset_sd=float(np.std(rs)),
                    asc=tie_asc,
                    pooling_effect=100.0 * (tie_pool_all - float(np.mean(ra))),
                    cellline_effect=100.0 * (tie_asc - float(np.mean(rs))))
    R("%-10s pooled %.4f | random-single %.4f (sd %.4f) | ASC %.4f || pooling %+.2f pts | cell line %+.2f pts"
      % (qid, tie_pool_all, np.mean(ra), np.std(ra), tie_asc,
         TIE[qid]["pooling_effect"], TIE[qid]["cellline_effect"]))
R("%d draws per query, %ds" % (NDRAW, round(time.time() - t0)))
# GUARD: the random-single draw must actually vary. If every draw returned the same rows the control
# would be silently degenerate and its agreement with ASC would mean nothing.
_a = np.array([r[rng.integers(len(r))] for r in ROWS]); _b = np.array([r[rng.integers(len(r))] for r in ROWS])
_multi = SIZES > 1
assert _multi.sum() > 1000, ("TOO FEW MULTI-SIGNATURE COMPOUNDS FOR THE CONTROL TO MEAN ANYTHING", int(_multi.sum()))
assert (_a[_multi] != _b[_multi]).mean() > 0.2, ("THE RANDOM-SINGLE DRAW IS NOT VARYING", float((_a[_multi] != _b[_multi]).mean()))
R("guard: two independent draws differ on %.1f%% of the %d multi-signature compounds"
  % (100 * (_a[_multi] != _b[_multi]).mean(), int(_multi.sum())))

In [ ]:
# CELL 6: the verdict against the criterion fixed in cell 1, and the report.
PE = np.array([TIE[q]["pooling_effect"] for q in QIDS])
CE = np.array([TIE[q]["cellline_effect"] for q in QIDS])
npos = int((PE > 0).sum()); medPE = float(np.median(PE))
MET = (npos >= NEED) and (medPE >= FLOOR)
R("--- POOLING EFFECT, pooled minus random-single, percentage points ---")
for q, v in zip(QIDS, PE): R("  %-10s %+.2f" % (q, v))
R("  positive in %d of %d | median %+.2f | range %+.2f to %+.2f" % (npos, len(QIDS), medPE, PE.min(), PE.max()))
R("--- CELL LINE EFFECT, ASC single minus random-single on the same compounds ---")
for q, v in zip(QIDS, CE): R("  %-10s %+.2f" % (q, v))
R("  median %+.2f | range %+.2f to %+.2f" % (float(np.median(CE)), CE.min(), CE.max()))
R("CRITERION (positive in >= %d of %d AND median >= +%.1f pts): %s" % (NEED, len(QIDS), FLOOR, "MET" if MET else "NOT MET"))
if not MET:
    R("  -> SECTION 4 OF RESULT_SSC_MEK_2026-10-02.md IS WITHDRAWN, NOT SOFTENED, and there is no")
    R("     benchmark addendum. The B3 observation is attributed to the cell line.")
else:
    R("  -> the pooling effect is established on %d queries and belongs to the benchmark record." % len(QIDS))
out = dict(benchmark_freeze=BM_SHA, sscmek_freeze=SSC_SHA, queries=QIDS, ndraw=NDRAW,
           need=NEED, floor=FLOOR, criterion_met=bool(MET),
           pooling_effect={q: float(v) for q, v in zip(QIDS, PE)},
           cellline_effect={q: float(v) for q, v in zip(QIDS, CE)}, tie=TIE)
bts = json.dumps(out, sort_keys=True, default=float).encode()
open("sscmek_C_pooling_result.json", "wb").write(bts)
assert open("sscmek_C_pooling_result.json", "rb").read() == bts, "RESULT READ-BACK MISMATCH"
print("=== PHASE C REPORT BEGIN ===")
print("\n".join(REPORT))
print("--- HEADLINE ---")
print("queries %d | pooling effect positive in %d, median %+.2f pts | cell line effect median %+.2f pts"
      % (len(QIDS), npos, medPE, float(np.median(CE))))
print("CRITERION:", "MET" if MET else "NOT MET")
print("sscmek_C_pooling_result.json sha256", hashlib.sha256(bts).hexdigest())
print("=== PHASE C REPORT END ===")